# Global Sensitivity C — sample_weight Locked Model

공식 A의 Stage 3.5 동결 파라미터와 Stage 3 확정 25개 Feature를 그대로 사용한다. Global Train 안의 SAMPID별 eligible Person-Period 행 수 역수만 각 CV fold의 fit에 전달한다. 새 hyperparameter search, Feature Selection, Test 접근은 하지 않는다.

- CV: `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)`, groups=`SAMPID`
- scoring: positive-class F1 · threshold: 0.5
- 결과 해석·parameter 변경은 사람이 담당한다.


## 1. 입력 및 동결 조건

공식 Global Train과 Stage 3.5 결과만 읽는다. Test DatasetBundle은 만들지 않는다.


In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

ROOT = Path(os.environ.get("KHUDA_PROJECT_ROOT", Path.cwd())).resolve()
while not (ROOT / "code").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("KHUDA_PROJECT_ROOT에 저장소 루트를 지정하거나 저장소 안에서 Notebook을 실행하세요.")
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if "code" in sys.modules and not hasattr(sys.modules["code"], "__path__"):
    del sys.modules["code"]

from code.contracts import DatasetBundle
from code.model.locked_sensitivity import (
    load_baseline_stage_3_5_summary,
    load_stage_3_5_locked_params,
    run_locked_model_oof,
    save_locked_model_artifacts,
    sensitivity_comparison_table,
)
from code.pipeline.audit import (
    attach_person_period_column,
    calculate_train_sample_weight,
    load_selected_feature_names,
)
from code.pipeline.saved_results import load_saved_global_train, select_bundle_features

RESULT_ROOT = ROOT / "data" / "result" / "baseline_42features"
DATASET_PATH = RESULT_ROOT / "datasets" / "global_dataset.parquet"
PERSON_PERIOD_PATH = RESULT_ROOT / "datasets" / "person_period.parquet"
SPLIT_PATH = RESULT_ROOT / "splits" / "split_ids.csv"
FEATURE_CONFIG = ROOT / "code" / "config" / "features.yaml"
MODEL_CONFIG = ROOT / "code" / "config" / "model_config.yaml"
STAGE_3_DIR = RESULT_ROOT / "modeling" / "stage_3"
STAGE_3_5_DIR = RESULT_ROOT / "modeling" / "stage_3_5"
SELECTED_FEATURES_PATH = STAGE_3_DIR / "selected_features.csv"
LOCKED_PARAMS_PATH = STAGE_3_5_DIR / "final_refined_params.json"
BASELINE_SUMMARY_PATH = STAGE_3_5_DIR / "final_tuning_summary.csv"
BASELINE_FOLD_PATH = STAGE_3_5_DIR / "final_fold_f1.json"
BASELINE_OOF_PATHS = {
    "logistic_regression": STAGE_3_5_DIR / "refined_logistic_regression_oof_predictions.parquet",
    "xgboost": STAGE_3_5_DIR / "refined_xgboost_oof_predictions.parquet",
}
COLORS = {"A Baseline": "#7a7a7a", "B n_prior_periods": "#0066cc", "C sample_weight": "#0066cc"}
LABELS = {"logistic_regression": "Logistic Regression", "xgboost": "XGBoost"}
plt.rcParams.update({
    "figure.facecolor": "#f5f5f7", "axes.facecolor": "#ffffff", "axes.edgecolor": "#e0e0e0",
    "text.color": "#1d1d1f", "axes.labelcolor": "#1d1d1f", "xtick.color": "#1d1d1f", "ytick.color": "#1d1d1f",
})


In [ ]:
# 기존 Global Train과 25개 공식 Feature만 사용한다. n_prior_periods는 predictor에 포함하지 않는다.
required_paths = [DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG, MODEL_CONFIG, SELECTED_FEATURES_PATH, LOCKED_PARAMS_PATH, BASELINE_SUMMARY_PATH, BASELINE_FOLD_PATH, *BASELINE_OOF_PATHS.values()]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("필요한 저장 산출물이 없습니다:" + chr(10) + chr(10).join(missing_paths))

base_train = load_saved_global_train(DATASET_PATH, SPLIT_PATH, FEATURE_CONFIG)
selected_features = load_selected_feature_names(SELECTED_FEATURES_PATH)
train_bundle = select_bundle_features(base_train, selected_features, name="global_sensitivity_sample_weight")
sample_weight = calculate_train_sample_weight(train_bundle.groups)
locked_params = load_stage_3_5_locked_params(LOCKED_PARAMS_PATH)
baseline_summary = load_baseline_stage_3_5_summary(BASELINE_SUMMARY_PATH)
assert len(selected_features) == 25
assert train_bundle.X.shape[1] == 25
assert "n_prior_periods" not in train_bundle.X.columns
assert sample_weight.gt(0).all() and sample_weight.le(1).all()
assert sample_weight.groupby(train_bundle.groups).sum().sub(1.0).abs().le(1e-12).all()
display(pd.DataFrame([{"strategy": "C sample_weight", "train_rows": len(train_bundle.y), "train_unique_SAMPID": train_bundle.groups.nunique(), "feature_count": train_bundle.X.shape[1], "sample_weight_used_for_fit": True, "test_used": False}]))
display(pd.DataFrame({"feature_order": range(1, 26), "feature": selected_features}))
display(pd.DataFrame({"model": list(locked_params), "locked_stage_3_5_params": list(locked_params.values())}))


## 2. Locked-parameter OOF

동결된 LR/XGBoost parameter를 각 CV fold의 fit에 그대로 전달한다. 탐색 함수는 호출하지 않는다.


In [ ]:
# sample_weight는 각 fold의 fit 행에만 전달한다. validation 및 OOF 지표는 비가중으로 계산된다.
locked_results = [
    run_locked_model_oof(train_bundle, strategy="C sample_weight", model_name=model, locked_params=locked_params[model], feature_config=FEATURE_CONFIG, model_config=MODEL_CONFIG, sample_weight_train=sample_weight)
    for model in ("logistic_regression", "xgboost")
]


## 3. Sensitivity 전용 결과 저장

A 공식 artifact는 덮어쓰지 않는다.


In [ ]:
# A 공식 결과와 분리된 sensitivity 전용 폴더에만 저장한다.
OUTPUT_DIR = RESULT_ROOT / "modeling" / "sensitivity_sample_weight"
artifact_paths = save_locked_model_artifacts(locked_results, train_bundle, OUTPUT_DIR)
display(pd.DataFrame({"artifact": list(artifact_paths), "path": [str(path) for path in artifact_paths.values()]}))


## 4. A Baseline과 비교

아래 표와 그래프는 사람이 비교·판단하기 위한 표시이며, 자동 선택이나 해석을 수행하지 않는다.


In [ ]:
STRATEGY_LABEL = "C sample_weight"
# A 공식 결과와 이번 Strategy의 결과만 비교한다. 결과를 해석하거나 parameter를 변경하지 않는다.
comparison = sensitivity_comparison_table(baseline_summary, locked_results)
display(comparison)

with BASELINE_FOLD_PATH.open(encoding="utf-8") as file:
    baseline_fold_f1 = json.load(file)["stage_3_5"]
baseline_oof = {model: pd.read_parquet(path) for model, path in BASELINE_OOF_PATHS.items()}
locked_oof = {result.model: result.oof_predictions for result in locked_results}
locked_summary = pd.DataFrame([result.summary_row() for result in locked_results])

# 1. A vs Strategy CV F1 mean ± std
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(2)
width = 0.34
baseline_plot = baseline_summary.set_index("model").loc[["logistic_regression", "xgboost"]]
strategy_plot = locked_summary.set_index("model").loc[["logistic_regression", "xgboost"]]
ax.bar(x - width / 2, baseline_plot["cv_f1_mean"], width, yerr=baseline_plot["cv_f1_std"], capsize=5, color=COLORS["A Baseline"], label="A Baseline")
ax.bar(x + width / 2, strategy_plot["cv_f1_mean"], width, yerr=strategy_plot["cv_f1_std"], capsize=5, color=COLORS[STRATEGY_LABEL], label=STRATEGY_LABEL)
ax.set_xticks(x, [LABELS[name] for name in baseline_plot.index])
ax.set_ylabel("CV F1 mean ± std")
ax.set_title(f"A Baseline vs {STRATEGY_LABEL}: CV F1")
ax.legend()
plt.tight_layout()
plt.show()

# 2. A vs Strategy 5-fold F1
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, model in zip(axes, ("logistic_regression", "xgboost")):
    ax.plot(range(1, 6), baseline_fold_f1[model], marker="o", color=COLORS["A Baseline"], label="A Baseline")
    ax.plot(range(1, 6), next(result.fold_f1 for result in locked_results if result.model == model), marker="o", color=COLORS[STRATEGY_LABEL], label=STRATEGY_LABEL)
    ax.set_title(f"{LABELS[model]} fold F1")
    ax.set_xlabel("CV fold")
    ax.set_xticks(range(1, 6))
    ax.legend()
axes[0].set_ylabel("F1")
fig.suptitle(f"A Baseline vs {STRATEGY_LABEL}: 5-fold F1", y=1.03)
plt.tight_layout()
plt.show()

# 3. A vs Strategy OOF Precision / Recall / F1
metrics = ["oof_precision", "oof_recall", "oof_f1"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, model in zip(axes, ("logistic_regression", "xgboost")):
    baseline_values = baseline_plot.loc[model, metrics]
    strategy_values = strategy_plot.loc[model, metrics]
    positions = np.arange(len(metrics))
    ax.bar(positions - width / 2, baseline_values, width, color=COLORS["A Baseline"], label="A Baseline")
    ax.bar(positions + width / 2, strategy_values, width, color=COLORS[STRATEGY_LABEL], label=STRATEGY_LABEL)
    ax.set_title(LABELS[model])
    ax.set_xticks(positions, ["Precision", "Recall", "F1"])
    ax.legend()
axes[0].set_ylabel("OOF metric")
fig.suptitle(f"A Baseline vs {STRATEGY_LABEL}: OOF classification metrics", y=1.03)
plt.tight_layout()
plt.show()

# 4–5. LR / XGBoost OOF ROC curves
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model in zip(axes, ("logistic_regression", "xgboost")):
    base = baseline_oof[model]
    current = locked_oof[model]
    RocCurveDisplay.from_predictions(base["y_true"], base["y_proba"], name="A Baseline", color=COLORS["A Baseline"], ax=ax)
    RocCurveDisplay.from_predictions(current["y_true"], current["y_probability"], name=STRATEGY_LABEL, color=COLORS[STRATEGY_LABEL], ax=ax)
    ax.set_title(f"{LABELS[model]} OOF ROC")
fig.suptitle(f"A Baseline vs {STRATEGY_LABEL}: OOF ROC curves", y=1.02)
plt.tight_layout()
plt.show()

# 6. LR / XGBoost confusion matrices (A and this Strategy)
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for row, model in enumerate(("logistic_regression", "xgboost")):
    base = baseline_oof[model]
    current = locked_oof[model]
    ConfusionMatrixDisplay.from_predictions(base["y_true"], base["y_pred_at_0_5"], ax=axes[row, 0], colorbar=False)
    axes[row, 0].set_title(f"{LABELS[model]}: A Baseline")
    ConfusionMatrixDisplay.from_predictions(current["y_true"], current["y_predicted"], ax=axes[row, 1], colorbar=False)
    axes[row, 1].set_title(f"{LABELS[model]}: {STRATEGY_LABEL}")
fig.suptitle(f"A Baseline vs {STRATEGY_LABEL}: OOF confusion matrices", y=1.01)
plt.tight_layout()
plt.show()
